[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C19_Bayesian_ML_Course/03_variational_inference/03_variational_inference.ipynb)

# 03 · 变分推断 VI（纯 numpy 从零）

目标：从零实现 **ELBO**、**平均场 CAVI**、**重参数化梯度 VI**，把近似后验与**解析后验 / MCMC** 对拍；并验证 **ELBO 单调上升**这一 VI 的头号自检。

路线：两高斯 KL 闭式 → ELBO 与证据的关系 → 平均场 CAVI 解高斯均值+精度（对拍解析/MCMC）→ ELBO 单调性 → 重参数化梯度 VI（对拍解析后验）→ mode-seeking 演示 → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊。

> 心智模型：**VI = 从简单分布族里挑最像后验的那个**。最大化 ELBO ⟺ 最小化 KL(q‖后验)。用 KL(q‖p) 导致 **mode-seeking、低估方差**——这是 VI 最该记住的偏差。

## 1 · 两个高斯的 KL 散度（闭式）

VI 处处要算 KL。两个一维高斯的 KL 有简洁闭式：

$$ \mathrm{KL}(\mathcal N(\mu_q,\sigma_q^2)\,\Vert\,\mathcal N(\mu_p,\sigma_p^2)) = \log\frac{\sigma_p}{\sigma_q} + \frac{\sigma_q^2 + (\mu_q-\mu_p)^2}{2\sigma_p^2} - \frac12 $$

验证：① KL≥0；② 自己对自己 KL=0；③ 对拍蒙特卡洛估计。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def check_allclose(name, got, ref, atol=1e-8, rtol=1e-5):
    got = np.asarray(got, float); ref = np.asarray(ref, float)
    ok = np.allclose(got, ref, atol=atol, rtol=rtol)
    err = float(np.max(np.abs(got - ref))) if got.size else 0.0
    print(f'[{name:<32}] allclose={ok}  max|err|={err:.2e}')
    assert ok, f'{name} 不一致'
    return ok

def kl_gaussian(mq, sq, mp, sp):
    '''KL( N(mq,sq^2) || N(mp,sp^2) )，闭式。'''
    return np.log(sp / sq) + (sq ** 2 + (mq - mp) ** 2) / (2 * sp ** 2) - 0.5

# 自己对自己 = 0
check_allclose('KL(p||p)=0', kl_gaussian(1.0, 2.0, 1.0, 2.0), 0.0, atol=1e-12)
# 非负
assert kl_gaussian(0.0, 1.0, 3.0, 2.0) > 0
# 对拍蒙特卡洛: E_q[log q - log p]
mq, sq, mp, sp = 0.5, 0.8, 1.5, 1.2
z = rng.normal(mq, sq, size=2000000)
logq = -0.5*np.log(2*np.pi*sq**2) - 0.5*(z-mq)**2/sq**2
logp = -0.5*np.log(2*np.pi*sp**2) - 0.5*(z-mp)**2/sp**2
check_allclose('KL 闭式 vs MC', kl_gaussian(mq,sq,mp,sp), (logq-logp).mean(), atol=3e-3)
print('✅ 两高斯 KL 闭式正确（VI 的常用工具）')

## 2 · ELBO + KL = log 证据（在共轭模型上精确验证）

核心恒等式：$\log p(D)=\mathcal L(q)+\mathrm{KL}(q\Vert p(\theta\mid D))$。

在共轭 Beta-Binomial 上一切可算：真后验、$\log p(D)$（边际似然有闭式）、任意 $q$ 的 ELBO 都能精确算，验证三者满足上式。这把抽象的 ELBO 落到可对拍的数。

In [ ]:
from math import lgamma
def log_beta_fn(a, b):
    return lgamma(a) + lgamma(b) - lgamma(a + b)

# 模型: theta~Beta(a0,b0), k~Binomial(n,theta). 取一个 q=Beta(aq,bq)（非最优也行）
a0, b0, n, k = 2.0, 2.0, 10, 7
# 边际似然 p(D) 的闭式: C(n,k) * B(a0+k,b0+n-k)/B(a0,b0)
log_evidence = (lgamma(n+1)-lgamma(k+1)-lgamma(n-k+1)) + log_beta_fn(a0+k, b0+n-k) - log_beta_fn(a0, b0)
# 真后验 Beta(a0+k, b0+n-k)
ap, bp = a0 + k, b0 + (n - k)

def elbo_beta(aq, bq, grid=None):
    '''ELBO = E_q[log p(theta,D)] - E_q[log q]，用网格积分精确算。'''
    g = np.linspace(1e-6, 1-1e-6, 200001)
    logq = (aq-1)*np.log(g) + (bq-1)*np.log1p(-g) - log_beta_fn(aq, bq)
    q = np.exp(logq)
    log_joint = ((a0-1)*np.log(g)+(b0-1)*np.log1p(-g)-log_beta_fn(a0,b0)  # log prior
                 + k*np.log(g)+(n-k)*np.log1p(-g)                          # log lik(∝)
                 + (lgamma(n+1)-lgamma(k+1)-lgamma(n-k+1)))                # 组合数常数
    integrand = q * (log_joint - logq)
    return np.trapezoid(integrand, g) if hasattr(np,'trapezoid') else np.trapz(integrand, g)

def kl_beta(aq, bq):
    g = np.linspace(1e-6, 1-1e-6, 200001)
    logq = (aq-1)*np.log(g)+(bq-1)*np.log1p(-g)-log_beta_fn(aq,bq)
    logpost = (ap-1)*np.log(g)+(bp-1)*np.log1p(-g)-log_beta_fn(ap,bp)
    q = np.exp(logq)
    integrand = q*(logq-logpost)
    return np.trapezoid(integrand, g) if hasattr(np,'trapezoid') else np.trapz(integrand, g)

# 取一个任意 q
aq, bq = 5.0, 3.0
L = elbo_beta(aq, bq); KL = kl_beta(aq, bq)
print(f'ELBO={L:.5f}, KL={KL:.5f}, ELBO+KL={L+KL:.5f}, log p(D)={log_evidence:.5f}')
check_allclose('ELBO + KL == log evidence', L + KL, log_evidence, atol=1e-3)
# q=真后验时 KL=0, ELBO=log evidence（最优）
check_allclose('q=后验时 ELBO==log evidence', elbo_beta(ap, bp), log_evidence, atol=1e-3)
check_allclose('q=后验时 KL==0', kl_beta(ap, bp), 0.0, atol=1e-4)
print('✅ ELBO+KL=log证据；q=真后验时 KL=0、ELBO 达到上界 log p(D)')

## 3 · 平均场 CAVI：高斯均值 μ + 精度 τ（对拍 MCMC）

模型同模块 02 §5：$x_i\sim\mathcal N(\mu,1/\tau)$，先验 $\mu\sim\mathcal N(\mu_0,1/\tau_0)$、$\tau\sim\mathrm{Gamma}(a_0,b_0)$。

平均场 $q(\mu,\tau)=q(\mu)q(\tau)$。推导 CAVI 更新（$q(\mu)$ 是高斯、$q(\tau)$ 是 Gamma）：
- $q(\mu)=\mathcal N(m,1/p)$：$p=\tau_0+n\,\mathbb{E}[\tau]$，$m=(\tau_0\mu_0+\mathbb{E}[\tau]\,n\bar x)/p$；
- $q(\tau)=\mathrm{Gamma}(a_0+n/2,\ b_0+\tfrac12\mathbb{E}_\mu[\sum(x_i-\mu)^2])$，
  其中 $\mathbb{E}_\mu[\sum(x_i-\mu)^2]=\sum(x_i-m)^2+n/p$（用到 $\mathrm{Var}_q[\mu]=1/p$）。

用期望 $\mathbb{E}[\tau]=a/b$（Gamma 均值）。对拍模块 02 的 Gibbs 后验。

In [ ]:
def cavi_normal(data, mu0, tau0, a0, b0, n_iter=100):
    x = np.asarray(data, float); n = len(x); xbar = x.mean()
    # 初始化 q(tau)=Gamma(a_q,b_q), q(mu)=N(m,1/p)
    a_q, b_q = a0 + n/2.0, b0 + n*np.var(x)/2.0   # b_q 暂用样本方差初始化
    m, p = xbar, 1.0
    for _ in range(n_iter):
        E_tau = a_q / b_q
        # update q(mu)
        p = tau0 + n * E_tau
        m = (tau0 * mu0 + E_tau * n * xbar) / p
        # update q(tau): E_mu[Σ(x-mu)^2] = Σ(x-m)^2 + n*Var[mu], Var[mu]=1/p
        E_sq = np.sum((x - m) ** 2) + n / p
        a_q = a0 + n / 2.0
        b_q = b0 + 0.5 * E_sq
    return dict(mu_mean=m, mu_var=1.0/p, tau_a=a_q, tau_b=b_q, E_tau=a_q/b_q)

data = rng.normal(5.0, 2.0, size=50)
res = cavi_normal(data, mu0=0.0, tau0=0.01, a0=2.0, b0=2.0, n_iter=200)
print(f"CAVI: E[mu]={res['mu_mean']:.3f} (数据均值={data.mean():.3f}), E[tau]={res['E_tau']:.3f}")

# 对拍 Gibbs（重跑模块 02 的采样器）
def gibbs_normal(data, mu0, tau0, a0, b0, n_samp, rng, burn=2000):
    x=np.asarray(data,float); n=len(x); xbar=x.mean(); mu,tau=xbar,1.0; out=np.empty((n_samp,2))
    for t in range(n_samp+burn):
        pr=tau0+n*tau; mu=rng.normal((tau0*mu0+tau*n*xbar)/pr, np.sqrt(1/pr))
        tau=rng.gamma(a0+n/2, 1.0/(b0+0.5*np.sum((x-mu)**2)))
        if t>=burn: out[t-burn]=(mu,tau)
    return out
S = gibbs_normal(data, 0.0, 0.01, 2.0, 2.0, 60000, np.random.default_rng(1))
print(f'Gibbs: E[mu]={S[:,0].mean():.3f}, E[tau]={S[:,1].mean():.3f}')
check_allclose('CAVI E[mu] vs Gibbs', res['mu_mean'], S[:,0].mean(), atol=0.1)
check_allclose('CAVI E[tau] vs Gibbs', res['E_tau'], S[:,1].mean(), atol=0.05)
print('✅ 平均场 CAVI 的后验均值 ≈ Gibbs（MCMC）—— 两条路殊途同归')

## 4 · ELBO 必须单调上升（VI 的头号自检）

CAVI 的每次因子更新都（弱）单调抬高 ELBO。把每轮 ELBO 打出来，**若它下降必有 bug**。这是 VI 实现最重要的调试工具。我们算 Normal 模型的 ELBO 并验证整条曲线单调不降。

In [ ]:
def elbo_normal(data, mu0, tau0, a0, b0, m, p, a_q, b_q):
    '''该平均场 q 下的 ELBO = E_q[log p(x,mu,tau)] - E_q[log q]。'''
    x = np.asarray(data, float); n = len(x)
    from scipy_free_digamma import digamma  # 见下：自带 digamma
    E_tau = a_q / b_q
    E_log_tau = digamma(a_q) - np.log(b_q)         # E[log tau], tau~Gamma
    E_sq = np.sum((x - m) ** 2) + n / p            # E_mu[Σ(x-mu)^2]
    # E[log lik] = Σ [0.5 E_log_tau - 0.5 log2pi - 0.5 E_tau (x-mu)^2]
    E_loglik = 0.5*n*E_log_tau - 0.5*n*np.log(2*np.pi) - 0.5*E_tau*E_sq
    # E[log prior mu] = 0.5 log(tau0/2pi) - 0.5 tau0 E[(mu-mu0)^2]; E[(mu-mu0)^2]=(m-mu0)^2+1/p
    E_logp_mu = 0.5*np.log(tau0/(2*np.pi)) - 0.5*tau0*((m-mu0)**2 + 1/p)
    # E[log prior tau] = a0 log b0 - lgamma(a0) + (a0-1)E_log_tau - b0 E_tau
    E_logp_tau = a0*np.log(b0) - lgamma(a0) + (a0-1)*E_log_tau - b0*E_tau
    # entropy of q(mu)=N: 0.5 log(2pi e / p)
    H_mu = 0.5*np.log(2*np.pi*np.e/p)
    # -E[log q(tau)] for Gamma = entropy: a_q - log b_q + lgamma(a_q) + (1-a_q)digamma(a_q)
    H_tau = a_q - np.log(b_q) + lgamma(a_q) + (1-a_q)*digamma(a_q)
    return E_loglik + E_logp_mu + E_logp_tau + H_mu + H_tau

# 自带 digamma（避免依赖 scipy）：用 numpy 实现
import sys, types
_dg = types.ModuleType('scipy_free_digamma')
def _digamma(x):
    # 渐近 + 递推，x>0 标量足够精确
    x = float(x); result = 0.0
    while x < 6:
        result -= 1.0/x; x += 1.0
    f = 1.0/(x*x)
    result += np.log(x) + 0.5/x - f*(1/12.0 - f*(1/120.0 - f*(1/252.0)))
    return result
_dg.digamma = _digamma
sys.modules['scipy_free_digamma'] = _dg
from scipy_free_digamma import digamma

# 跑 CAVI 并记录每轮 ELBO
x = data; n=len(x); xbar=x.mean()
mu0,tau0,a0,b0 = 0.0,0.01,2.0,2.0
a_q,b_q = a0+n/2, b0+n*np.var(x)/2; m,pp = xbar,1.0
elbos=[]
for _ in range(60):
    E_tau=a_q/b_q; pp=tau0+n*E_tau; m=(tau0*mu0+E_tau*n*xbar)/pp
    E_sq=np.sum((x-m)**2)+n/pp; a_q=a0+n/2; b_q=b0+0.5*E_sq
    elbos.append(elbo_normal(x,mu0,tau0,a0,b0,m,pp,a_q,b_q))
elbos=np.array(elbos)
diffs = np.diff(elbos)
print(f'ELBO 从 {elbos[0]:.3f} 升到 {elbos[-1]:.3f}')
assert np.all(diffs > -1e-6), 'ELBO 必须单调不降！下降=有bug'
print('✅ ELBO 单调上升 —— CAVI 实现正确（这是 VI 的头号自检）')

## 5 · 重参数化梯度 VI（对拍解析后验）

黑盒 VI：用 $\theta=\mu+\sigma\epsilon$（$\epsilon\sim\mathcal N(0,1)$）让梯度穿过采样，对**任意**可微 log 联合密度直接最大化 ELBO，无需手推 CAVI。

在共轭 Normal-Normal 上测（已知方差，推断均值 μ），真后验是高斯、有解析解，验证 VI 学到的 $q$ 收敛到它。

In [ ]:
# 模型: mu~N(0, 5^2), x_i~N(mu, sigma^2 known). 真后验解析（模块01）
sigma = 2.0
data5 = rng.normal(3.0, sigma, size=30)
mu0, tau0_sq = 0.0, 25.0
n5 = len(data5); xbar5 = data5.mean()
post_var = 1.0/(1.0/tau0_sq + n5/sigma**2)
post_mean = post_var*(mu0/tau0_sq + n5*xbar5/sigma**2)
print(f'解析后验: N({post_mean:.4f}, {post_var:.4f})')

def log_joint(mu):
    loglik = np.sum(-0.5*(data5-mu)**2/sigma**2)
    logprior = -0.5*mu**2/tau0_sq
    return loglik + logprior

def reparam_vi(log_joint_fn, n_iter=4000, lr=0.02, n_mc=20, seed=0):
    '''q=N(m, exp(log_s)^2). 重参数化 + 蒙特卡洛梯度上升 ELBO。'''
    rng_v = np.random.default_rng(seed)
    m, log_s = 0.0, 0.0                          # 优化 log_s 保证 sigma>0
    for it in range(n_iter):
        s = np.exp(log_s)
        eps = rng_v.standard_normal(n_mc)
        theta = m + s*eps                        # 重参数化采样
        # ELBO = E[log_joint(theta)] + entropy(q);  entropy(N)=log_s+const
        # 对 theta 的梯度（数值/解析）：grad_theta log_joint
        # 解析: d/dmu log_joint = Σ(x-mu)/sigma^2 - mu/tau0_sq
        g_theta = np.sum(data5[:,None]-theta[None,:], axis=0)/sigma**2 - theta/tau0_sq
        # 链式: dtheta/dm=1, dtheta/dlog_s = s*eps
        grad_m = np.mean(g_theta * 1.0)
        grad_log_s = np.mean(g_theta * (s*eps)) + 1.0   # +1 来自 entropy d(log_s)/d(log_s)
        m += lr*grad_m
        log_s += lr*grad_log_s
    return m, np.exp(log_s)

m_vi, s_vi = reparam_vi(log_joint, n_iter=6000, lr=0.01, n_mc=50)
print(f'VI 学到: N({m_vi:.4f}, {s_vi**2:.4f})')
check_allclose('VI 后验均值 vs 解析', m_vi, post_mean, atol=0.05)
check_allclose('VI 后验方差 vs 解析', s_vi**2, post_var, atol=0.03)
print('✅ 重参数化梯度 VI 收敛到解析后验（共轭高斯上 KL 间隙=0）')

## 6 · mode-seeking：VI 低估方差、漏峰

用 KL(q‖p) 的 VI 在**双峰**目标上只会抓**一个峰**（mode-seeking），而 MCMC 会探索两个峰。这是 VI 最该记住的偏差。我们用一个双峰目标直观展示。

In [ ]:
# 双峰目标: 0.5 N(-3,0.6^2) + 0.5 N(3,0.6^2)
def log_bimodal(x):
    x = np.atleast_1d(x)
    c1 = np.exp(-0.5*(x+3)**2/0.6**2)
    c2 = np.exp(-0.5*(x-3)**2/0.6**2)
    return np.log(0.5*c1 + 0.5*c2 + 1e-300)

# 单高斯 q 的 VI：从某初值出发会收敛到其中一个峰（而非覆盖两峰）
def vi_single_gaussian(log_target, m_init, n_iter=3000, lr=0.01, n_mc=100, seed=0):
    rng_v=np.random.default_rng(seed); m, log_s = float(m_init), 0.0
    for _ in range(n_iter):
        s=np.exp(log_s); eps=rng_v.standard_normal(n_mc); th=m+s*eps
        # 数值梯度 d log_target/dtheta
        h=1e-4; g=(log_target(th+h)-log_target(th-h))/(2*h)
        m += lr*np.mean(g); log_s += lr*(np.mean(g*s*eps)+1.0)
    return m, np.exp(log_s)

m_pos,_ = vi_single_gaussian(log_bimodal, m_init=2.0, seed=1)
m_neg,_ = vi_single_gaussian(log_bimodal, m_init=-2.0, seed=2)
print(f'VI 从右侧初始化 -> 抓住峰 {m_pos:.2f}; 从左侧 -> 抓住峰 {m_neg:.2f}')
assert m_pos > 1.5 and m_neg < -1.5, 'VI 各自收敛到一个峰（mode-seeking）'
# 对比 MCMC（用模块02的 MH）会在两峰间跳
def _scalar(v):
    return float(np.ravel(np.asarray(v))[0])   # 取标量（兼容返回1元数组的 log_target）
def mh1d(logt, x0, n, step, rng, burn=1000):
    x=float(x0); lp=_scalar(logt(x)); out=[]
    for t in range(n+burn):
        xp=x+step*rng.standard_normal(); lpp=_scalar(logt(xp))
        if np.log(rng.uniform())<lpp-lp: x,lp=xp,lpp
        if t>=burn: out.append(x)
    return np.array(out)
mc = mh1d(log_bimodal, 0.0, 60000, 4.0, np.random.default_rng(3))
frac_left = np.mean(mc < 0)
print(f'MCMC 落在左峰的比例={frac_left:.2f}（≈0.5，覆盖两峰）')
assert 0.3 < frac_left < 0.7, 'MCMC 应大致均匀覆盖两峰'
print('✅ VI mode-seeking 只抓一峰；MCMC 覆盖两峰 —— VI 的核心偏差')

---
## ✏️ 练习 1：高斯 KL 散度

实现 `kl_gaussian(mq, sq, mp, sp)`（一维），验证非负、自对自为 0、且方向不对称（$\mathrm{KL}(p\Vert q)\ne\mathrm{KL}(q\Vert p)$）。

In [ ]:
def kl_gaussian(mq, sq, mp, sp):
    # TODO: log(sp/sq) + (sq^2 + (mq-mp)^2)/(2 sp^2) - 1/2
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert abs(kl_gaussian(1.0, 2.0, 1.0, 2.0)) < 1e-12, '自对自=0'
assert kl_gaussian(0.0, 1.0, 2.0, 1.0) > 0, 'KL 非负'
# 不对称
f = kl_gaussian(0.0, 1.0, 2.0, 3.0)
b = kl_gaussian(2.0, 3.0, 0.0, 1.0)
assert abs(f - b) > 1e-6, 'KL 一般不对称'
# 均值差越大 KL 越大
assert kl_gaussian(0,1,5,1) > kl_gaussian(0,1,1,1)
print('✅ 练习 1 通过：高斯 KL 闭式、非负、不对称')

## ✏️ 练习 2：ELBO 分解

ELBO 有等价写法 $\mathcal L=\mathbb{E}_q[\log p(D\mid\theta)]-\mathrm{KL}(q(\theta)\Vert p(\theta))$（拟合 − 偏离先验）。实现 `elbo_decomposed(E_loglik, kl_q_prior)` 返回 ELBO，并验证它 = 「期望对数似然 − KL(q‖先验)」。（这是 VAE 损失 = 重构 − KL 的来源。）

In [ ]:
def elbo_decomposed(E_loglik, kl_q_prior):
    # TODO: ELBO = E_loglik - kl_q_prior
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 拟合越好(E_loglik 越大) ELBO 越大; 越偏离先验(KL 越大) ELBO 越小
assert elbo_decomposed(-10.0, 2.0) == -12.0
assert elbo_decomposed(-5.0, 2.0) > elbo_decomposed(-10.0, 2.0), '拟合更好 -> ELBO 更大'
assert elbo_decomposed(-5.0, 1.0) > elbo_decomposed(-5.0, 5.0), '更贴近先验 -> ELBO 更大'
# q=先验时 KL=0, ELBO=E_loglik
assert elbo_decomposed(-7.0, 0.0) == -7.0
print('✅ 练习 2 通过：ELBO = 拟合数据 − 偏离先验（VAE 损失的来源）')

## ✏️ 练习 3：CAVI 更新 q(μ)

在 Normal 均值+精度模型里，给定当前 $\mathbb{E}[\tau]$，实现 $q(\mu)$ 的 CAVI 更新 `cavi_update_mu(xbar, n, mu0, tau0, E_tau)`，返回后验 `(m, var)`：
$p=\tau_0+n\mathbb{E}[\tau]$，$m=(\tau_0\mu_0+\mathbb{E}[\tau]\,n\bar x)/p$，$\mathrm{var}=1/p$。

In [ ]:
def cavi_update_mu(xbar, n, mu0, tau0, E_tau):
    # TODO: 精度相加 + 精度加权（用 E_tau 作数据精度）
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
m, v = cavi_update_mu(xbar=5.0, n=10, mu0=0.0, tau0=1.0, E_tau=0.5)
# 精度 = 1 + 10*0.5 = 6 -> var=1/6
assert abs(v - 1/6) < 1e-9
assert abs(m - (1.0*0.0 + 0.5*10*5.0)/6.0) < 1e-9
# E_tau 越大（数据越可信）-> m 越靠近 xbar
m_hi,_ = cavi_update_mu(5.0, 10, 0.0, 1.0, 10.0)
m_lo,_ = cavi_update_mu(5.0, 10, 0.0, 1.0, 0.01)
assert abs(m_hi - 5.0) < abs(m_lo - 5.0), '数据精度越高 -> 越靠近样本均值'
print('✅ 练习 3 通过：CAVI 的 q(μ) 更新 = 精度加权（与 Gibbs 同公式，但用期望）')

## ✏️ 练习 4：监控 ELBO 单调性

实现 `is_monotonic_increasing(elbos, tol=1e-6)`：检查一串 ELBO 是否单调不降（允许 tol 的数值噪声）。这是判断 CAVI 实现是否正确的关键自检。

In [ ]:
def is_monotonic_increasing(elbos, tol=1e-6):
    # TODO: 相邻差 >= -tol 全部成立则 True
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert is_monotonic_increasing([1.0, 2.0, 2.5, 2.5, 3.0]) == True
assert is_monotonic_increasing([1.0, 2.0, 1.5]) == False, '下降应被抓出'
# 允许微小数值噪声
assert is_monotonic_increasing([1.0, 1.0 - 1e-9, 2.0], tol=1e-6) == True
# 真实 CAVI 的 elbos 应通过
assert is_monotonic_increasing(list(elbos)) == True, '前面 CAVI 的 ELBO 应单调'
print('✅ 练习 4 通过：ELBO 单调性自检 —— VI 最重要的调试工具')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def kl_gaussian(mq, sq, mp, sp):
    return np.log(sp/sq) + (sq**2 + (mq-mp)**2)/(2*sp**2) - 0.5

In [ ]:
# 练习 2 参考答案
def elbo_decomposed(E_loglik, kl_q_prior):
    return E_loglik - kl_q_prior

In [ ]:
# 练习 3 参考答案
def cavi_update_mu(xbar, n, mu0, tau0, E_tau):
    p = tau0 + n * E_tau
    m = (tau0 * mu0 + E_tau * n * xbar) / p
    return m, 1.0 / p

In [ ]:
# 练习 4 参考答案
def is_monotonic_increasing(elbos, tol=1e-6):
    e = np.asarray(elbos, float)
    return bool(np.all(np.diff(e) >= -tol))

---
## 🧪 真实数据胶囊：变分高斯混合（鸢尾花 Iris 的无监督聚类）

真实任务：用**变分 / EM** 拟合高斯混合模型（GMM）对经典 **Iris** 数据聚类（真实数据，3 类）。GMM 的后验（聚类分配 + 各类参数）没有闭式解，正是变分方法的主场。

这里实现 GMM 的 **EM**（它是 VI 在『后验对参数取点估计』时的特例，且 E 步的责任度 = 平均场 $q$ 对隐变量的近似），验证：① 对数似然单调上升（与 ELBO 单调同理）；② 多次随机重启取对数似然最高者，聚类与真实标签高度吻合。带 try/except 回退到内置 Iris 真实数值（全 4 维）。

In [ ]:
# Iris 数据（真实 4 维特征; 内置真实数值, 无需联网/sklearn）
try:
    from sklearn.datasets import load_iris
    _d = load_iris(); X_iris = _d.data; y_iris = _d.target
    print('用 sklearn 加载 Iris（150×4）')
except Exception:
    print('无 sklearn，用内置 Iris 真实数值（4 维, 每类 12 个真实样本）')
    # Iris 真实测量值 [sepal_len, sepal_wid, petal_len, petal_wid]，每类 12 个真实样本
    setosa = [[5.1,3.5,1.4,0.2],[4.9,3.0,1.4,0.2],[4.7,3.2,1.3,0.2],[4.6,3.1,1.5,0.2],
              [5.0,3.6,1.4,0.2],[5.4,3.9,1.7,0.4],[4.6,3.4,1.4,0.3],[5.0,3.4,1.5,0.2],
              [4.4,2.9,1.4,0.2],[4.9,3.1,1.5,0.1],[5.4,3.7,1.5,0.2],[4.8,3.4,1.6,0.2]]
    versi = [[7.0,3.2,4.7,1.4],[6.4,3.2,4.5,1.5],[6.9,3.1,4.9,1.5],[5.5,2.3,4.0,1.3],
             [6.5,2.8,4.6,1.5],[5.7,2.8,4.5,1.3],[6.3,3.3,4.7,1.6],[4.9,2.4,3.3,1.0],
             [6.6,2.9,4.6,1.3],[5.2,2.7,3.9,1.4],[5.9,3.0,4.2,1.5],[6.0,2.2,4.0,1.0]]
    virg = [[6.3,3.3,6.0,2.5],[5.8,2.7,5.1,1.9],[7.1,3.0,5.9,2.1],[6.3,2.9,5.6,1.8],
            [6.5,3.0,5.8,2.2],[7.6,3.0,6.6,2.1],[7.3,2.9,6.3,1.8],[6.7,2.5,5.8,1.8],
            [7.2,3.6,6.1,2.5],[6.5,3.2,5.1,2.0],[6.4,2.7,5.3,1.9],[6.8,3.0,5.5,2.1]]
    X_iris = np.array(setosa+versi+virg)
    y_iris = np.array([0]*12+[1]*12+[2]*12)
print('Iris 数据形状:', X_iris.shape)

In [ ]:
def gmm_em(X, K, n_iter=100, seed=0):
    '''GMM EM。返回 (责任度 resp, 每轮对数似然 lls, 聚类标签)。'''
    rng_g = np.random.default_rng(seed)
    N, D = X.shape
    # 初始化：随机选 K 个点当均值
    means = X[rng_g.choice(N, K, replace=False)].copy()
    covs = np.array([np.cov(X.T) for _ in range(K)])
    weights = np.ones(K) / K
    lls = []
    def gauss_logpdf(x, m, S):
        D = len(m); diff = x - m
        L = np.linalg.cholesky(S + 1e-6*np.eye(D))
        sol = np.linalg.solve(L, diff.T).T
        logdet = 2*np.sum(np.log(np.diag(L)))
        return -0.5*(D*np.log(2*np.pi) + logdet + np.sum(sol**2, axis=1))
    for _ in range(n_iter):
        # E 步：责任度（平均场 q 对隐分配的近似）
        log_r = np.array([np.log(weights[k]) + gauss_logpdf(X, means[k], covs[k]) for k in range(K)]).T
        mx = log_r.max(axis=1, keepdims=True)
        ll = np.sum(mx.ravel() + np.log(np.sum(np.exp(log_r - mx), axis=1)))
        lls.append(ll)
        resp = np.exp(log_r - mx); resp /= resp.sum(axis=1, keepdims=True)
        # M 步
        Nk = resp.sum(axis=0)
        weights = Nk / N
        means = (resp.T @ X) / Nk[:, None]
        for k in range(K):
            diff = X - means[k]
            covs[k] = (resp[:, k:k+1] * diff).T @ diff / Nk[k]
    return resp, np.array(lls), resp.argmax(axis=1)

# EM 是非凸的，多次随机重启取对数似然最高者（标准实践）
best = None
for seed in range(8):
    resp_s, lls_s, lab_s = gmm_em(X_iris, K=3, n_iter=80, seed=seed)
    assert np.all(np.diff(lls_s) > -1e-6), 'EM 对数似然必须单调不降'   # 每次重启都单调
    if best is None or lls_s[-1] > best[0]:
        best = (lls_s[-1], lls_s, lab_s)
lls, labels = best[1], best[2]
print(f'最佳重启: 对数似然从 {lls[0]:.1f} 升到 {lls[-1]:.1f} (每次重启都单调 ✅)')

# 聚类与真实标签的吻合度（用最佳标签匹配算 purity）
def purity(pred, true):
    total = 0
    for c in np.unique(pred):
        mask = pred == c
        total += np.bincount(true[mask]).max()
    return total / len(true)
pur = purity(labels, y_iris)
print(f'聚类纯度 = {pur:.2%}')
assert pur > 0.8, 'GMM（最佳重启）应较好地还原 Iris 的 3 类结构'
print('✅ 胶囊验证：变分/EM 拟合 GMM，对数似然单调、聚类还原真实结构')

**🧪 胶囊练习**：实现 `responsibilities(X, means, covs, weights, gauss_logpdf)`：GMM 的 E 步，返回每个点属于各类的责任度（平均场 $q$ 对隐变量分配的近似），每行和为 1。用 log-sum-exp 保稳定。

In [ ]:
def responsibilities(X, means, covs, weights, gauss_logpdf):
    # TODO: log_r[n,k] = log w_k + gauss_logpdf(X, means[k], covs[k]);
    #       行 softmax（log-sum-exp）归一化
    raise NotImplementedError

In [ ]:
# 自测
def gauss_logpdf(x, m, S):
    D=len(m); diff=x-m; L=np.linalg.cholesky(S+1e-6*np.eye(D))
    sol=np.linalg.solve(L, diff.T).T; logdet=2*np.sum(np.log(np.diag(L)))
    return -0.5*(D*np.log(2*np.pi)+logdet+np.sum(sol**2,axis=1))
means = X_iris[[0, 20, 40]]; covs = np.array([np.cov(X_iris.T)]*3); weights = np.ones(3)/3
r = responsibilities(X_iris, means, covs, weights, gauss_logpdf)
assert r.shape == (len(X_iris), 3)
assert np.allclose(r.sum(axis=1), 1.0), '每行责任度和为 1'
assert np.all(r >= 0), '责任度非负'
print('✅ 胶囊练习通过：E 步责任度 = 平均场 q 对隐分配的近似')

In [ ]:
# 📖 胶囊参考答案
def responsibilities(X, means, covs, weights, gauss_logpdf):
    K = len(weights)
    log_r = np.array([np.log(weights[k]) + gauss_logpdf(X, means[k], covs[k]) for k in range(K)]).T
    mx = log_r.max(axis=1, keepdims=True)
    r = np.exp(log_r - mx)
    return r / r.sum(axis=1, keepdims=True)

### 小结
- **VI 把推断变成优化**：从简单分布族 $\mathcal Q$ 里挑最像后验的 $q$，最小化 $\mathrm{KL}(q\Vert$后验$)$。
- **ELBO**：$\log p(D)=\mathcal L(q)+\mathrm{KL}$，因 KL≥0 故 $\mathcal L$ 是下界；**最大化 ELBO ⟺ 最小化 KL**（避开了算不出的后验）。等价分解 = 拟合数据 − 偏离先验（VAE 损失）。
- **平均场** $q=\prod q_j$ 假设独立 → 可分解优化，但**丢相关、低估方差**（KL(q‖p) 的 mode-seeking）。
- **CAVI**：$\log q_j^*=\mathbb{E}_{q_{-j}}[\log p(\theta,D)]$，与 Gibbs **对偶**（取期望 vs 采样），ELBO 单调上升。
- **重参数化** $\theta=\mu+\sigma\epsilon$ 让梯度穿过采样 → 黑盒/可扩展 VI（VAE 的核心）。
- **VI vs MCMC**：VI 快但有偏、低估不确定性；MCMC 慢但渐近无偏。按问题选，或杂交。

**下一站**：**模块 04 · 高斯过程** —— 把贝叶斯推断搬到无限维的函数空间，自动给出不确定性。